# ImpactLens: train on Kaggle (free GPU)

Trains the two models on the packed datasets: a Siamese U-Net for change (OSCD) and a plain U-Net for flood water (Sen1Floods11).

**Before you run it** (right-hand *Notebook options* / *Session options* panel):
1. **Accelerator**: GPU (T4 x2 or P100).
2. **Internet**: On (needed to `git clone`; Kaggle asks you to verify your phone number once).
3. **Add data**: your private dataset `impactlens-packed` (the two `.npz` files from `scripts/pack_for_kaggle.py`).
4. `REPO_URL` below points at the public GitHub repo (change it if you forked).

Evaluation rule: checkpoints are chosen on *validation* data only (OSCD: 3 held-out train cities; Sen1Floods11: its `valid` split). The test splits are scored once, at the end.

In [ ]:
# ---- settings you may edit -------------------------------------------------------------
REPO_URL = "https://github.com/bhavishyabondalapati/impactlens.git"  # public repo
BRANCH = "main"
WORK = "/kaggle/working"

## 1. Get the code

In [ ]:
import os
import subprocess

REPO_DIR = f"{WORK}/impactlens"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
print(
    subprocess.run(
        ["git", "-C", REPO_DIR, "log", "--oneline", "-3"], capture_output=True, text=True
    ).stdout
)

Install the package **without dependencies**: Kaggle already ships a CUDA build of PyTorch and numpy,
and we must not let pip replace it. Packed data needs no rasterio, so nothing else is required.
`--ignore-requires-python` because the repo pins Python 3.11 and Kaggle's Python may differ.
`onnx` and `onnxruntime` are small and needed for the ONNX export check at the end of training.

In [ ]:
!pip install -q --no-deps --ignore-requires-python -e {REPO_DIR}
!pip install -q onnx onnxruntime
import sys

# A fresh editable install is not always visible to an already-running kernel; make sure it is.
sys.path.insert(0, f"{REPO_DIR}/src")
import impactlens

print("impactlens", impactlens.__version__)

## 2. Check the GPU and find the data

In [ ]:
import glob

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Turn on an accelerator in the session options.")

hits = glob.glob("/kaggle/input/**/oscd.npz", recursive=True)
assert hits, "oscd.npz not found: add your impactlens-packed dataset via 'Add data'"
DATA = os.path.dirname(hits[0])
print("data dir:", DATA)
for f in sorted(os.listdir(DATA)):
    print(f"  {f:24} {os.path.getsize(os.path.join(DATA, f)) / 1e6:8.1f} MB")

## 3. Smoke-test the data
Loads every split through the real dataset classes. If a count is off, stop here and re-pack.

In [ ]:
from impactlens.data.oscd import OSCDDataset
from impactlens.data.sen1floods11 import Sen1Floods11Dataset

oscd_npz, s1f_npz = f"{DATA}/oscd.npz", f"{DATA}/sen1floods11.npz"
for split in ("train", "val", "test"):
    print(f"OSCD {split:5}: {len(OSCDDataset(oscd_npz, split)):4} fixed tiles")
print(
    "OSCD train random crops per epoch: e.g.",
    len(OSCDDataset(oscd_npz, "train", crops_per_epoch=500)),
)
for split in ("train", "valid", "test", "bolivia"):
    print(f"Sen1Floods11 {split:8}: {len(Sen1Floods11Dataset(s1f_npz, split)):4} tiles")

item = OSCDDataset(oscd_npz, "train", crops_per_epoch=1, augment=True)[0]
print({k: (tuple(v.shape), round(float(v.mean()), 3)) for k, v in item.items() if k != "city"})

## 4. Train both models
Two runs, one after the other (`train.py --task oscd` then `--task floods`). Each one:
trains with early stopping on the **validation** split, scores the held-out test split once with the
best checkpoint, and exports + checks an ONNX file. The floods run also scores the NDWI baseline.
Everything is written to `/kaggle/working/oscd` and `/kaggle/working/floods`
(`best.pt`, `<task>.onnx`, `metrics.json`), which Kaggle keeps when you press **Save Version**.
Expect roughly 10-30 minutes per task on a T4; lower `--epochs` for a quick trial.

In [ ]:
import subprocess


def run(cmd):
    """Run a command and stream its output into the notebook; raise if it fails."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


for task in ("oscd", "floods"):
    run(
        [
            "python",
            "-m",
            "impactlens.models.train",
            "--task",
            task,
            "--data",
            DATA,
            "--out",
            f"{WORK}/{task}",
            "--workers",
            "2",
        ]
    )

## 5. Results
Prints validation vs test numbers (and the NDWI baseline for floods), then lists the output files.
Download them from the notebook's **Output** tab (or `kaggle kernels output <user>/<notebook> -p results/`).

In [ ]:
import json

for task in ("oscd", "floods"):
    m = json.load(open(f"{WORK}/{task}/metrics.json"))
    print(f"== {task}: best epoch {m['best_epoch']} of {len(m['history'])} ==")
    for split in ("val", "test", "bolivia"):
        if split in m:
            r = m[split]
            print(
                f"  {split:8} IoU {r['iou']:.3f}  F1 {r['f1']:.3f}  "
                f"P {r['precision']:.3f}  R {r['recall']:.3f}"
            )
    if "baseline_ndwi" in m:
        b = m["baseline_ndwi"]
        for split in ("val", "test", "bolivia"):
            print(
                f"  NDWI baseline {split:8} F1 {b[split]['fixed']['f1']:.3f} (thr 0)  "
                f"{b[split]['tuned']['f1']:.3f} (tuned on val)"
            )
    print("  ONNX ok:", m["onnx"]["ok"], "| max logit diff", m["onnx"]["max_abs_logit_diff"])

for task in ("oscd", "floods"):
    for f in sorted(os.listdir(f"{WORK}/{task}")):
        print(f"{os.path.getsize(f'{WORK}/{task}/{f}') / 1e6:8.2f} MB  {WORK}/{task}/{f}")